# Support Vector Regression (SVR)

Support Vector Regression (SVR) is the regression variant of the Support Vector Machine (SVM). Instead of finding a hyperplane that separates classes, SVR finds a function that fits the training data within a specified margin of tolerance — the **epsilon-tube** — while remaining as flat (simple) as possible. Data points that fall inside the tube contribute no loss; only those outside it are penalized. This makes SVR robust to outliers and particularly effective for non-linear regression problems via the **kernel trick**.

## 1. Key Concepts

Given a training set $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$, SVR aims to find a function $f(\mathbf{x}) = \mathbf{w}^\top \phi(\mathbf{x}) + b$ such that:

$$|y_i - f(\mathbf{x}_i)| \leq \varepsilon \quad \text{for all } i$$

Where $\varepsilon$ (epsilon) defines the half-width of the **insensitive tube** around the predicted function. Points within the tube are ignored; points outside are penalized via **slack variables** $\xi_i, \xi_i^*$:

$$\xi_i = \max(0,\ y_i - f(\mathbf{x}_i) - \varepsilon), \qquad \xi_i^* = \max(0,\ f(\mathbf{x}_i) - y_i - \varepsilon)$$

The optimization problem balances flatness of $f$ (minimizing $\|\mathbf{w}\|^2$) against tolerance of deviations beyond $\varepsilon$:

$$\min_{\mathbf{w}, b, \xi, \xi^*} \quad \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_{i=1}^{N} (\xi_i + \xi_i^*)$$
$$\text{subject to} \quad y_i - \mathbf{w}^\top \phi(\mathbf{x}_i) - b \leq \varepsilon + \xi_i, \quad \mathbf{w}^\top \phi(\mathbf{x}_i) + b - y_i \leq \varepsilon + \xi_i^*, \quad \xi_i, \xi_i^* \geq 0$$

The parameter **C** controls this trade-off: a large $C$ penalizes violations heavily (less tolerance, risk of overfitting), while a small $C$ allows more violations in exchange for a simpler model.

## 2. Dual Formulation and the Kernel Trick

Solving the primal problem directly requires working in the (potentially infinite-dimensional) feature space $\phi(\mathbf{x})$. The **dual formulation** sidesteps this by expressing the solution in terms of inner products between training samples:

$$\min_{\alpha, \alpha^*} \quad \frac{1}{2}(\alpha - \alpha^*)^\top K (\alpha - \alpha^*) + \varepsilon \sum_i (\alpha_i + \alpha_i^*) - \sum_i y_i (\alpha_i - \alpha_i^*)$$
$$\text{subject to} \quad \sum_i (\alpha_i - \alpha_i^*) = 0, \qquad 0 \leq \alpha_i, \alpha_i^* \leq C$$

Where $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^\top \phi(\mathbf{x}_j)$ is the **kernel matrix** — computed entirely without ever explicitly computing $\phi$.

The prediction for a new point $\mathbf{x}$ is then:

$$\hat{y} = \sum_{i=1}^{N} (\alpha_i - \alpha_i^*) \, k(\mathbf{x}_i, \mathbf{x}) + b$$

Only samples with $\alpha_i \neq 0$ or $\alpha_i^* \neq 0$ contribute to the prediction — these are the **support vectors**, typically a small subset of the training data.

## 3. Available Kernels

The kernel function $k(\mathbf{x}_i, \mathbf{x}_j)$ implicitly maps the inputs to a higher-dimensional space, allowing SVR to model non-linear relationships. Four kernels are supported:

| Kernel | Formula | Parameters | Best used when |
|--------|---------|------------|----------------|
| Linear (`lin`) | $\mathbf{x}_i^\top \mathbf{x}_j$ | — | Data is linearly separable or high-dimensional |
| Polynomial (`poly`) | $(\mathbf{x}_i^\top \mathbf{x}_j + c)^d$ | $c$, $d$ | Moderate non-linearity, known degree of interaction |
| RBF (`rbf`) | $\exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$ | $\gamma$ | General-purpose, unknown structure, smooth functions |
| Sigmoid (`sig`) | $\tanh(\alpha\, \mathbf{x}_i^\top \mathbf{x}_j + c)$ | $\alpha$, $c$ | Neural-network-like behavior |

**In practice, RBF is the default choice** for most regression problems — it requires only one parameter ($\gamma$) and can approximate any continuous function.

## 4. Bias Estimation

After solving the dual problem and obtaining $(\alpha, \alpha^*)$, the bias term $b$ is estimated from the **support vectors** — training points that lie on or outside the $\varepsilon$-tube, i.e., those with $\alpha_i > 10^{-6}$ or $\alpha_i^* > 10^{-6}$:

$$b = \frac{1}{|\mathcal{SV}|} \sum_{i \in \mathcal{SV}} \left( y_i - \sum_{j=1}^{N} (\alpha_j - \alpha_j^*) k(\mathbf{x}_j, \mathbf{x}_i) \right)$$

If no support vectors are found (rare edge case), the bias falls back to the mean residual over all training points.

**Note on numerical stability**: The implementation adds a small regularization term $10^{-6} \mathbf{I}$ to the kernel matrix before solving ($K_{\text{reg}} = K + 10^{-6} \mathbf{I}$) to ensure it remains positive semi-definite, preventing numerical issues in the convex solver.

## 5. Pseudo-algorithm

**Training phase** — `fit(X, y)`:

1. $\mathcal{D} \leftarrow$ training set $\{(\mathbf{x}_i, y_i)\}_{i=1}^N$, kernel $k$, parameters $C$, $\varepsilon$
2. Compute kernel matrix $K_{ij} \leftarrow k(\mathbf{x}_i, \mathbf{x}_j)$ for all $i, j$
3. $K_{\text{reg}} \leftarrow K + 10^{-6} \mathbf{I}$ (numerical stabilization)
4. Solve dual QP: $(\alpha^*, \alpha^{*\star}) \leftarrow \arg\min$ dual objective subject to constraints
5. $\mathcal{SV} \leftarrow \{i : |\alpha_i| > 10^{-6}$ or $|\alpha_i^*| > 10^{-6}\}$
6. $b \leftarrow \text{mean}_{i \in \mathcal{SV}} \left( y_i - \sum_j (\alpha_j - \alpha_j^*) K_{ji} \right)$
7. **return** $(\alpha^*, \alpha^{*\star}, b)$

**Prediction phase** — `predict(X_test)`:

8. Compute cross-kernel $K_{\text{test},ij} \leftarrow k(\mathbf{x}_i^{\text{train}}, \mathbf{x}_j^{\text{test}})$
9. $\hat{\mathbf{y}} \leftarrow K_{\text{test}}^\top (\alpha - \alpha^*) + b$
10. **return** $\hat{\mathbf{y}}$

## 6. Hyperparameters Summary

| Parameter | Role | Effect of increasing |
|-----------|------|---------------------|
| `C_reg` | Regularization strength | Less tolerance for errors → tighter fit, risk of overfitting |
| `epsilon` | Half-width of the insensitive tube | Fewer support vectors → smoother, simpler model |
| `gamma` (RBF) | Influence radius of a single training point | Smaller neighborhoods → more complex, risk of overfitting |
| `d` (poly) | Degree of the polynomial kernel | Higher non-linearity → more expressive, harder to optimize |
| `alpha` (sigmoid) | Scaling of the dot product | Steeper sigmoid → sharper transitions |

## 7. Implementation

For this implementation, we use the **Boston Housing** equivalent — the **California Housing** dataset — focusing on a single feature (`MedInc`, median income) for easier visualization of the fitted curve alongside the full multi-feature comparison against Scikit-learn.

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter, Scaler

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target)

# SVR is sensitive to feature scale — standardization is mandatory
scaler = Scaler(method='standard')
X_scaled = scaler.fit_transform(X)

In [ ]:
X.head()

In [ ]:
# Split the data
splitter = DataSplitter(seed=42)
X_train, X_test, y_train, y_test = splitter.train_test_split(X_scaled, y, test_size=0.2)

### With Scikit-learn

In [ ]:
import time

In [ ]:
from sklearn.svm import SVR as SklearnSVR

start_1 = time.perf_counter()
sk_svr = SklearnSVR(kernel='rbf', C=1.0, epsilon=0.1, gamma=1.0)
sk_svr.fit(X_train, y_train)
end_1 = time.perf_counter()

### With ifri-mini-ml-lib

In [ ]:
from ifri_mini_ml_lib.regression import SVR

start_2 = time.perf_counter()
my_svr = SVR(C_reg=1.0, epsilon=0.1, kernel='rbf', gamma=1.0)
my_svr.fit(X_train.values, y_train.values)
end_2 = time.perf_counter()

In [ ]:
# Predict on the test set
y_pred_1 = sk_svr.predict(X_test)
y_pred_2 = my_svr.predict(X_test.values)

In [ ]:
# Evaluate both models
from ifri_mini_ml_lib.metrics.regression import mean_squared_error, mean_absolute_error, r2_score

mse_1 = mean_squared_error(y_test, y_pred_1)
mse_2 = mean_squared_error(y_test, y_pred_2)

mae_1 = mean_absolute_error(y_test, y_pred_1)
mae_2 = mean_absolute_error(y_test, y_pred_2)

r2_1  = r2_score(y_test, y_pred_1)
r2_2  = r2_score(y_test, y_pred_2)

In [ ]:
# Summary table
results = pd.DataFrame({
    'Metric': ['MSE', 'MAE', 'R² Score', 'Training Time (s)'],
    'Scikit-learn': [mse_1, mae_1, r2_1, end_1 - start_1],
    'ifri-mini-ml-lib': [mse_2, mae_2, r2_2, end_2 - start_2],
})

results.T

## 8. Interactive Demo

In [ ]:
from ipywidgets import interact, FloatSlider, Dropdown
from notebooks.regression.utils import plot_svr

interact(
    plot_svr,
    kernel=Dropdown(
        options=[('Linear', 'lin'), ('Polynomial', 'poly'), ('RBF', 'rbf'), ('Sigmoid', 'sig')],
        value='rbf',
        description='Kernel:'
    ),
    C_reg=FloatSlider(min=0.01, max=10.0, step=0.1, value=1.0, description='C:'),
    epsilon=FloatSlider(min=0.01, max=1.0, step=0.05, value=0.1, description='Epsilon:'),
    gamma=FloatSlider(min=0.01, max=5.0, step=0.1, value=1.0, description='Gamma:'),
);

The interactive demo lets you explore the effect of each hyperparameter on the fitted SVR curve. Increasing `C` forces the model to minimize errors more aggressively, producing a tighter fit. Increasing `epsilon` widens the insensitive tube — fewer points become support vectors and the model becomes smoother. With the RBF kernel, a high `gamma` creates sharp, localized fits while a low `gamma` produces a smoother, more global response.

## 9. Real-life Applications

SVR is particularly valued in domains where robustness to outliers and the ability to model complex non-linear patterns are critical.

1. **Financial Time Series Forecasting**: SVR has been widely applied to predict stock prices, exchange rates, and asset returns. Its $\varepsilon$-insensitive loss makes it more robust to noisy market data than least-squares methods.

2. **Load and Energy Forecasting**: Power grid operators use SVR to predict electricity demand based on temperature, time of day, and historical consumption. The RBF kernel captures the non-linear seasonal and daily patterns effectively.

3. **Bioinformatics**: SVR is used to predict continuous biological quantities such as protein structure properties, gene expression levels, and drug-target binding affinities from molecular feature descriptors.

4. **Environmental Modeling**: SVR models air quality indices, water pollution levels, and climate variables from sensor readings, where non-linearities are common and training data is limited.

## 10. Limitations and Challenges

Despite its theoretical elegance and strong empirical performance, SVR has practical limitations worth knowing.

1. **Scalability**: Solving the dual QP problem has a time complexity of approximately $\mathcal{O}(N^2)$ to $\mathcal{O}(N^3)$ in the number of training samples. This makes SVR impractical for large datasets (typically beyond ~50,000 samples) without approximation methods.

2. **Sensitivity to Hyperparameters**: The choice of kernel, $C$, $\varepsilon$, and kernel-specific parameters ($\gamma$, $d$, etc.) heavily influences performance. SVR typically requires careful grid search or Bayesian optimization, which compounds the computational cost.

3. **Feature Scaling Requirement**: SVR is not scale-invariant. Distance-based kernels (RBF in particular) give disproportionate weight to features with large magnitudes. **Standardization is mandatory** before training.

4. **No Probabilistic Output**: Unlike Gaussian Process Regression, SVR produces point predictions only — no confidence intervals or uncertainty estimates are provided natively.

These limitations motivate the use of approximations such as **Nyström kernel methods** for scalability, or alternative models like **Gaussian Processes** when uncertainty quantification is needed.

## 11. References

- Support Vector Regression, Scikit-learn User Guide, https://scikit-learn.org/stable/modules/svm.html#svr
- Smola, A. J. & Schölkopf, B. (2004). *A Tutorial on Support Vector Regression*. Statistics and Computing, 14, 199–222.
- Vapnik, V. (1995). *The Nature of Statistical Learning Theory*. Springer.
- Support Vector Machines, StatQuest with Josh Starmer, https://www.youtube.com/watch?v=efR1C6CvhmE
- CVXPY Documentation, https://www.cvxpy.org/examples/machine_learning/svr.html